![image.png](https://i.imgur.com/a3uAqnb.png)

# **⚡ Wind Turbine Object Detection with RT-DETR**
In this lab, we will:
✅ **Use RT-DETR** for **wind turbine detection**  
✅ **Understand the dataset structure**  
✅ **Train a RT-DETR model**  
✅ **Evaluate the model on the validation set**  
✅ **Run inference on test images**  

---

## **1️⃣ Understanding the Dataset Structure**
The dataset follows the **RT-DETR format**, which consists of:
📂 **train/** → Training images & labels  
📂 **valid/** → Validation images & labels  
📂 **test/** → Test images (for inference)  
📜 **data.yaml** → Defines dataset paths & class names  

### **🔹 DETR Dataset Folder Structure**
```
drone_dataset/
│── train/
│   │── images/
│   │   ├── pic_031.jpg
│   │   ├── pic_032.jpg
│   │   ├── ...
│   │── labels/
│   │   ├── pic_031.txt
│   │   ├── pic_032.txt
│   │   ├── ...
│
│── valid/
│   │── images/
│   │   ├── pic_035.jpg
│   │   ├── pic_036.jpg
│   │   ├── ...
│   │── labels/
│   │   ├── pic_035.txt
│   │   ├── pic_036.txt
│   │   ├── ...
│
│── test/
│   │── images/
│   │   ├── pic_040.jpg
│   │   ├── pic_041.jpg
│   │   ├── ...
│   │── labels/
│   │   ├── pic_040.txt
│   │   ├── pic_041.txt
│   │   ├── ...
│
│── data.yaml
```
Each **image** has a **corresponding label** file with the **same name**, but a `.txt` extension.

---

## **2️⃣ What’s Inside a RT-DETR Label File?**
Each `.txt` file contains **annotations** in this format:

```
<class_id> <x_center> <y_center> <width> <height>
```

✅ **All values are normalized** between **0 and 1**  
✅ The **bounding box** is defined by its **center** and **size**  

### **🔹 Example (`pic_031.txt`)**
```
0 0.526 0.448 0.12 0.15
1 0.731 0.602 0.18 0.22
```
- **First column** → Class ID (`0` = cable tower, `1` = turbine)  
- **Rest** → Bounding box (normalized)  

---

## **3️⃣ Loading the Dataset**

In [ ]:
import kagglehub

# Download the dataset
path = kagglehub.dataset_download("kylegraupe/wind-turbine-image-dataset-for-computer-vision")

print("Path to dataset files:", path)

In [ ]:
# Load dataset configuration
dataset_path = path + "/data.yaml"

# Check dataset information
print(open(dataset_path).read())

## **4️⃣ Training a DETR Model**
We will fine-tune a **pretrained DETR model**.

In [ ]:
# Install Ultralytics library which has RT-DETR
!pip install -q ultralytics

In [ ]:
from ultralytics import RTDETR
import matplotlib.pyplot as plt
# loads the model
model = RTDETR("rtdetr-l.pt") # <-- 'l' stands for the large model, the downloaded weights are trained on COCO

#### Let's try to predict an image before training

In [ ]:
# Load an image and run inference
results = model(path + "/test/images/windmill1_jpg.rf.35f47a69d0373596edb7578eea5151f6.jpg", save=True)

# Convert result to a NumPy array and display
predicted_image = results[0].plot()  # Convert prediction to an image

plt.figure(figsize=(16, 16))
plt.imshow(predicted_image)
plt.axis("off")
plt.title("Predicted Image")
plt.show()

#OMG, AI will take over the world.🫠

In [ ]:
import os
import urllib.request
from ultralytics import RTDETR

# an image that has something on the COCO dataset
image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image_name = "image_c59262.jpg"

print(f"Downloading COCO sample image as {image_name}...")
urllib.request.urlretrieve(image_url, image_name)
print("Download complete!")

In [ ]:
results = model(image_name)
predicted_image = results[0].plot()  # Convert prediction to an image

plt.figure(figsize=(16, 16))
plt.imshow(predicted_image)
plt.axis("off")
plt.title("Predicted Image")
plt.show()


##



#### Nevermind, let's train it! 🚀

In [ ]:
model.train(data=dataset_path, epochs=5, imgsz=640)

## **5️⃣ Evaluating the Model**
We use **mAP@0.5:0.95** to assess performance.

In [ ]:
# Run validation
metrics = model.val(data=dataset_path)

## **6️⃣ Running Inference on Test Images**


In [ ]:
# Ultralytics stores the best weights after training, we can load it later if we wanted
model = RTDETR("/content/runs/detect/train/weights/best.pt")

In [ ]:
# Load an image and run inference
results = model(path + "/valid/images/windmill19_jpg.rf.f8224e52590245a427cbdef03e618c48.jpg", save=True)

# Convert result to a NumPy array and display
predicted_image = results[0].plot()  # Convert prediction to an image

plt.figure(figsize=(8, 8))
plt.imshow(predicted_image)
plt.axis("off")
plt.title("Predicted Image")
plt.show()


### 🚀 **Now you have a working DETR object detection pipeline for wind turbines!**
![image.png](https://i.imgur.com/rGGLEsK.png)

### Contributed by: Abdulrahman Alfrihidi & Muhannad

### BONUS !
you already studied Tensorflow lite, ultralytics allows you to export your model to a TFlite format, which you can drop in to your microcontroller with ease !

In [ ]:
# export the model to a tflite format with quantization of 8 bits
model.eval()
model.export(format="litert", quantize='int8', data=dataset_path)
# wait why did we need the data to export a model ... ?

#### Model Deployment Summary

* **Model Parameters:** 31,987,850 (~32M)
* **Original Size:** 130 Mb
* **Quantized Size:** 35.5 Mb
* **Required RAM To Deploy:** ~36.5 Mb ~4x smaller size

**Note:** for smaller microcontrollers use smaller Ultralytics models.
> Now we can spot **Ali** everywhere !